In [ ]:
from pymongo import MongoClient
from datetime import datetime

# Conexão com o MongoDB (alterar o URI se necessário)
client = MongoClient("mongodb://localhost:27017/")

# Criando o banco de dados e a coleção
db = client["social_media_db"]
posts_collection = db["posts"]

# Função para inserir dados de postagens
def insert_post(platform, user, content, likes, comments, timestamp=None):
    if not timestamp:
        timestamp = datetime.now()
    post = {
        "platform": platform,
        "user": user,
        "content": content,
        "likes": likes,
        "comments": comments,
        "timestamp": timestamp
    }
    result = posts_collection.insert_one(post)
    print(f"Postagem inserida com ID: {result.inserted_id}")

# Função para analisar as postagens de uma plataforma específica
def analyze_platform(platform):
    print(f"\nAnálise de dados para a plataforma: {platform}")
    total_posts = posts_collection.count_documents({"platform": platform})
    avg_likes = posts_collection.aggregate([
        {"$match": {"platform": platform}},
        {"$group": {"_id": None, "average_likes": {"$avg": "$likes"}}}
    ])
    avg_comments = posts_collection.aggregate([
        {"$match": {"platform": platform}},
        {"$group": {"_id": None, "average_comments": {"$avg": "$comments"}}}
    ])

    print(f"Total de postagens: {total_posts}")
    for result in avg_likes:
        print(f"Média de likes: {result['average_likes']}")
    for result in avg_comments:
        print(f"Média de comentários: {result['average_comments']}")

# Função para buscar os posts mais populares
def top_posts(platform=None, limit=5):
    query = {"platform": platform} if platform else {}
    top_posts = posts_collection.find(query).sort("likes", -1).limit(limit)
    print("\nPosts mais populares:")
    for post in top_posts:
        print(f"Usuário: {post['user']}, Likes: {post['likes']}, Conteúdo: {post['content']}")

# Exemplo de uso
if __name__ == "__main__":
    # Inserindo postagens de exemplo
    insert_post("Twitter", "user1", "Adoro programar!", 120, 15)
    insert_post("Facebook", "user2", "Confira minhas férias!", 85, 20)
    insert_post("Instagram", "user3", "Nova receita no blog!", 200, 50)
    insert_post("Twitter", "user4", "Últimas notícias do mundo da tecnologia.", 150, 30)
    insert_post("Instagram", "user5", "Foto incrível de ontem!", 300, 100)

    # Analisando dados do Twitter
    analyze_platform("Twitter")

    # Buscando os posts mais populares
    top_posts(limit=3)
